In [3]:
import random

def quickselect(arr, k):
  if len(arr) == 1:
    return arr[0]

  pivote = random.choice(arr)
  menores = [x for x in arr if x < pivote]
  iguales = [x for x in arr if x == pivote]
  mayores = [x for x in arr if x > pivote]

  if k < len(menores):
    return quickselect(menores, k)
  elif k < len(menores) + len(iguales):
    return pivote
  else:
    return quickselect(mayores, k - len(menores) - len(iguales))


def median_quickselect(arr):
  n = len(arr)
  if n == 0:
    raise ValueError("ListaVacia")
  if n % 2 == 1:
    return quickselect(arr, n // 2)
  else:
    lo = quickselect(arr, n // 2 - 1)
    hi = quickselect(arr, n // 2)
    return (lo + hi) / 2


def _mediana_grupo(grupo):
  return sorted(grupo)[len(grupo) // 2]

def median_of_medians(arr, k):
  n = len(arr)
  if n == 0:
    raise ValueError("Lista vacia")
  if k < 0 or k >= n:
    raise IndexError("k fuera de rango")

  # Base case: if the array is small, sort and return the k-th element
  if n <= 5:
    return sorted(arr)[k]

  grupos = [arr[i:i+5] for i in range(0, n, 5)]
  medianas = [_mediana_grupo(g) for g in grupos]
  pivote = median_of_medians(medianas, len(medianas) // 2)
  menores = [x for x in arr if x < pivote]
  iguales = [x for x in arr if x == pivote]
  mayores = [x for x in arr if x > pivote]

  if k < len(menores):
    return median_of_medians(menores,k)
  elif k < len(menores) + len(iguales):
    return pivote
  else:
    return median_of_medians(mayores, k - len(menores) - len(iguales))



def median_por_muestreo(arr, delta = 0.05):
    n = len(arr)
    if n == 0:
        raise ValueError("Lista vacia")
    import math
    s = max(30, int(math.sqrt(n) * math.log( 1/ delta)*4))
    s = min(s,n)

    muestra = sorted(random.sample(arr,s))
    margen = max(1, int(math.sqrt(s) * math.log(1/ delta)))
    idx_lo = max(0, s // 2 - margen)
    idx_hi = min(s-1, s//2 + margen)
    lo, hi = muestra[idx_lo], muestra[idx_hi]

    rango = sorted(x for x in arr if lo <= x <= hi)
    rank_lo = sum(1 for x in arr if x < lo)
    objetivo = n// 2 - rank_lo

    if 0 <= objetivo < len(rango):
      if n % 2 == 1:
        return rango[objetivo]
      else:
        rank_total = rank_lo + objetivo
        if rank_total + 1 < len(rango) + rank_lo:
          hi_elem = rango[objetivo] if objetivo + 1 >= len(rango) \
                    else rango[objetivo + 1]

        else:
          hi_elem = rango[objetivo]
        return (rango[objetivo] + hi_elem) / 2

    else:
      return median_quickselect(arr)



def lazyselect(arr, k = None):
  import math
  n = len(arr)
  if k is None:
    k = n // 2

  if n <= 50:
    return sorted(arr)[k]

  while True:
    s_size = max(20, int(n ** 0.75))
    muestra = sorted(random.sample(arr,s_size))

    x = k * s_size / n
    delta = int(math.sqrt(n) * math.log(n))

    idx_l = max(0, int(x - math.sqrt(s_size)) - 1)
    idx_U = min(s_size-1, int(x + math.sqrt(s_size)) + 1)
    l, U = muestra[idx_l], muestra[idx_U]

    P = sorted(X for X in arr if l <= X <= U)
    rank_l = sum(1 for x in arr if x < l)
    rank_U = rank_l + len(P) - 1

    if rank_l <= k <= rank_U and len(P) <= 4 * s_size + 2:
      return P[k - rank_l]


if __name__ == "__main__":
    import time

    datos = random.sample(range(1, 10_001), 5000)
    mediana_real = sorted(datos)[len(datos) // 2]

    algoritmos = [
        ("Quickselect         O(n) esp.",  lambda a: median_quickselect(a)),
        ("Median of Medians   O(n) peor",  lambda a: median_of_medians(a, len(a)//2)),
        ("Muestreo aleatorio  O(√n·logn)", lambda a: median_por_muestreo(a)),
        ("LazySelect          ~1.5n comp.", lambda a: lazyselect(a)),
    ]

    print(f"n = {len(datos)}  |  Mediana real (índice n//2): {mediana_real}\n")
    print(f"{'Algoritmo':<38} {'Resultado':>10}  {'Tiempo':>10}")
    print("─" * 62)

    for nombre, fn in algoritmos:
        t0  = time.perf_counter()
        res = fn(list(datos))
        t1  = time.perf_counter()
        ok  = "✓" if res == mediana_real else "≈"
        print(f"{nombre:<38} {str(res):>10}  {(t1-t0)*1000:>8.3f} ms  {ok}")

n = 5000  |  Mediana real (índice n//2): 5026

Algoritmo                               Resultado      Tiempo
──────────────────────────────────────────────────────────────
Quickselect         O(n) esp.              5025.5     3.314 ms  ≈
Median of Medians   O(n) peor                5026     4.595 ms  ✓
Muestreo aleatorio  O(√n·logn)             5028.0     1.144 ms  ≈
LazySelect          ~1.5n comp.              5026     0.890 ms  ✓
